# Splitter Tutorial

This tutorial demonstrates the stable public API of `pepbenchmark.splitter`: random splitting, factory-based construction, and splitting from existing clustering results.

In [1]:
from pepbenchmark.cluster.interfaces import UnifiedClusterResult
from pepbenchmark.splitter import (
    RandomSplitter,
    UnifiedResultSplitter,
    create_splitter,
    list_available_splitters,
)

print('Available splitters:', list_available_splitters(include_aliases=True))

2026-03-27 05:23:36 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: cdhit
2026-03-27 05:23:36 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: mmseqs2
2026-03-27 05:23:37 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: motif
2026-03-27 05:23:37 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: kmer
2026-03-27 05:23:37 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: similarity


/home/batchcom/assist/miniforge3/envs/pepbenchmark/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


2026-03-27 05:23:42 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: molecular
Available splitters: ['cdhit', 'cold', 'ecfp', 'hybrid', 'kmer', 'mmseqs', 'random', 'hydra', 'mmseqs2']


## 1. Random Split

In [2]:
sequences = [
    'AAAAAA', 'BBBBBB', 'CCCCCC', 'DDDDDD',
    'EEEEEE', 'FFFFFF', 'GGGGGG', 'HHHHHH', 'IIIIII', 'JJJJJJ',
]
labels = [1, 1, 0, 0, 1, 0, 1, 0, 1, 0]

random_splitter = RandomSplitter()
random_splits = random_splitter.get_split_indices(
    data=sequences,
    frac_train=0.8,
    frac_valid=0.1,
    frac_test=0.1,
    seed=42,
)

print({k: len(v) for k, v in random_splits.items()})
print(random_splits)

2026-03-27 05:23:42 | INFO     | RandomSplitter | RandomSplitter initialized
2026-03-27 05:23:42 | INFO     | pepbenchmark.splitter.random_splitter | Starting random split: data_size=10, frac_train=0.8, frac_valid=0.1, frac_test=0.1, seed=42
2026-03-27 05:23:42 | INFO     | pepbenchmark.splitter.random_splitter | Random split completed: Train=8, Valid=1, Test=1
{'train': 8, 'valid': 1, 'test': 1}
{'train': [8, 1, 5, 0, 7, 2, 9, 4], 'valid': [3], 'test': [6]}


## 2. Create a Splitter with the Factory

In [3]:
splitter = create_splitter('random')
factory_splits = splitter.get_split_indices(sequences, seed=123)
print({k: len(v) for k, v in factory_splits.items()})

2026-03-27 05:23:42 | INFO     | RandomSplitter | RandomSplitter initialized
2026-03-27 05:23:42 | INFO     | pepbenchmark.splitter.random_splitter | Starting random split: data_size=10, frac_train=0.8, frac_valid=0.1, frac_test=0.1, seed=123
2026-03-27 05:23:42 | INFO     | pepbenchmark.splitter.random_splitter | Random split completed: Train=8, Valid=1, Test=1
{'train': 8, 'valid': 1, 'test': 1}


## 3. Split from Existing Clustering Results (Recommended)

In [4]:
# Manually construct an example UnifiedClusterResult
cluster_result = UnifiedClusterResult(
    cluster_assignments={
        'cluster_0': [0, 1, 2],
        'cluster_1': [3, 4],
        'cluster_2': [5, 6],
        'cluster_3': [7, 8, 9],
    },
    total_clusters=4,
    total_sequences=10,
    algorithm='demo',
    parameters={'source': 'tutorial'},
)

splitter = UnifiedResultSplitter(random_seed=42)
cluster_splits = splitter.get_split_indices(
    data=sequences,
    cluster_result=cluster_result,
    frac_train=0.8,
    frac_valid=0.1,
    frac_test=0.1,
    cluster_distribution_strategy='size_aware',
    preserve_cluster_integrity=True,
)

print({k: len(v) for k, v in cluster_splits.items()})
print(cluster_splits)

2026-03-27 05:23:43 | INFO     | pepbenchmark.splitter.unified_result_splitter | Starting split from cluster result: 4 clusters, 10 sequences
2026-03-27 05:23:43 | INFO     | pepbenchmark.splitter.unified_result_splitter | Split completed:
2026-03-27 05:23:43 | INFO     | pepbenchmark.splitter.unified_result_splitter |   train: 0 sequences (0.0%)
2026-03-27 05:23:43 | INFO     | pepbenchmark.splitter.unified_result_splitter |   valid: 5 sequences (50.0%)
2026-03-27 05:23:43 | INFO     | pepbenchmark.splitter.unified_result_splitter |   test: 5 sequences (50.0%)
2026-03-27 05:23:43 | INFO     | pepbenchmark.splitter.unified_result_splitter | Cluster distribution:
2026-03-27 05:23:43 | INFO     | pepbenchmark.splitter.unified_result_splitter |   train: 0 clusters
2026-03-27 05:23:43 | INFO     | pepbenchmark.splitter.unified_result_splitter |   valid: 2 clusters
2026-03-27 05:23:43 | INFO     | pepbenchmark.splitter.unified_result_splitter |   test: 2 clusters
{'train': 0, 'valid': 5, 't

## 4. Repeated Splits / k-Fold

In [5]:
multi_splits = splitter.generate_multiple_splits(
    cluster_result=cluster_result,
    n_splits=3,
    frac_train=0.8,
    frac_valid=0.1,
    frac_test=0.1,
)

print(list(multi_splits.keys()))
print({name: {k: len(v) for k, v in res.items()} for name, res in multi_splits.items()})

2026-03-27 05:23:45 | INFO     | pepbenchmark.splitter.unified_result_splitter | Generating 3 splits from cluster result
2026-03-27 05:23:45 | INFO     | pepbenchmark.splitter.unified_result_splitter | Starting split from cluster result: 4 clusters, 10 sequences
2026-03-27 05:23:45 | INFO     | pepbenchmark.splitter.unified_result_splitter | Split completed:
2026-03-27 05:23:45 | INFO     | pepbenchmark.splitter.unified_result_splitter |   train: 10 sequences (100.0%)
2026-03-27 05:23:45 | INFO     | pepbenchmark.splitter.unified_result_splitter |   valid: 0 sequences (0.0%)
2026-03-27 05:23:45 | INFO     | pepbenchmark.splitter.unified_result_splitter |   test: 0 sequences (0.0%)
2026-03-27 05:23:45 | INFO     | pepbenchmark.splitter.unified_result_splitter | Cluster distribution:
2026-03-27 05:23:45 | INFO     | pepbenchmark.splitter.unified_result_splitter |   train: 4 clusters
2026-03-27 05:23:45 | INFO     | pepbenchmark.splitter.unified_result_splitter |   valid: 0 clusters
2026-

## Summary

- `RandomSplitter` is suitable for quick baselines
- `create_splitter()` is useful as a unified configuration entry point
- `UnifiedResultSplitter` is ideal for reusing an existing cluster result
- Prefer importing from the package-level `pepbenchmark.splitter` entry points